# Penalties

In Skchange, the detector's decision is shaped by a penalty, which is a non-negative number (or array, see below) that controls the trade-off between under- and over-segmentation. A low penalty makes the detector eager to accept changes, producing many segments, while a high penalty makes it conservative, producing few.

For change scores, savings, and transient scores combined with a scalar penalty, the penalty acts as a threshold: A candidate changepoint is accepted only when its score exceeds the penalty. For costs, the penalty is added to the segmentation objective, so accepting an extra change point has to reduce the total cost by at least the penalty. Either way, higher penalty means fewer accepted changes.

The example below reuses a univariate series of length 30 with a mean change at index 20:

In [ ]:
from skchange.datasets import generate_piecewise_normal_data
from skchange.detectors import SeededBinarySegmentation
from skchange.interval_scorers import CUSUM

X = generate_piecewise_normal_data(means=[0, 5], lengths=[20, 10], seed=1)

low_penalty_detector = SeededBinarySegmentation(CUSUM(), penalty=0.1)
high_penalty_detector = SeededBinarySegmentation(CUSUM(), penalty=5.0)

print("Low penalty changepoints: ", low_penalty_detector.fit_predict(X))
print("High penalty changepoints:", high_penalty_detector.fit_predict(X))

## Calibrating the penalty
Skchange computes a sensible default penalty for each detector, typically a value derived from a theoretical formula such as BIC. In practice, the main knob you turn is `penalty_scale`, which is a positive multiplier applied to the default. For fine control, most detectors also let you override the raw penalty value directly.

Instead of relying on a theoretical formula, you can calibrate the penalty from data. For example, you can pick a value that achieves a target false-positive rate on validation data. This gives detectors that behave consistently across datasets with different scales and noise levels, without hand-tuning `penalty_scale`. The [skchange.tuning](../../api_reference/tuning.rst) module provides utilities for this.

## Penalty arrays
Some detectors support *array* penalties, where `penalty[k]` specifies the penalty for a change affecting `k` features.
The purpose of array penalties is to allow different levels of penalisation depending on how many features in a multivariate time series are affected by a change.
Plenty of statistical research has shown that different levels of penalisation are necessary for optimal detection of changes in multivariate time series when the number of affected features is unknown.